# Narrowband vs Wideband FM

This notebook isolates the GMRS-style deviation comparison from the legacy audio demo. It compares narrowband (±2.5 kHz) and wideband (±5 kHz) FM in both spectrum and recovered audio.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from rf_utils import *
from IPython.display import Audio, Markdown, display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

%matplotlib widget

plt.rcParams.update({
    "figure.figsize": (12, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})


In [ ]:
VOICE_FILE = ROOT / "assets" / "local" / "my_voice.m4a"
WORK_FS = 96_000
PLAY_FS = 44_100

if VOICE_FILE.exists():
    raw_fs, raw_audio = load_audio(VOICE_FILE, normalize_audio=True)
    raw_audio = normalize(ensure_mono(raw_audio))
    raw_audio = raw_audio[: int(raw_fs * 5)]
    voice_work = normalize(resample_signal(raw_audio, raw_fs, WORK_FS))
    voice_play = normalize(resample_signal(raw_audio, raw_fs, PLAY_FS))
    print(f"Loaded {VOICE_FILE.name} at {raw_fs} Hz")
else:
    t_fallback = np.arange(0, 3.0, 1 / WORK_FS)
    voice_work = normalize(
        0.7 * np.sin(2 * np.pi * 220 * t_fallback)
        + 0.4 * np.sin(2 * np.pi * 440 * t_fallback)
        + 0.2 * np.sin(2 * np.pi * 880 * t_fallback)
    )
    voice_play = normalize(resample_signal(voice_work, WORK_FS, PLAY_FS))
    print(f"No local voice recording found at {VOICE_FILE}. Using a synthetic fallback.")

t_work = np.arange(len(voice_work)) / WORK_FS


## Same Message, Different Deviation

Wider deviation improves recovered audio quality and noise performance, but it occupies more spectrum. That is the central tradeoff between narrowband and wideband FM.

In [ ]:
message = signal.sosfilt(signal.butter(5, [300, 3000], btype="band", fs=WORK_FS, output="sos"), voice_work)
message = normalize(message)
carrier_freq = 20_000

fm_narrow = fm_modulate(message, carrier_freq=carrier_freq, fs=WORK_FS, freq_dev=2_500)
fm_wide = fm_modulate(message, carrier_freq=carrier_freq, fs=WORK_FS, freq_dev=5_000)
demod_narrow = fm_demodulate(fm_narrow, fs=WORK_FS)
demod_wide = fm_demodulate(fm_wide, fs=WORK_FS)

fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
plot_spectrum(fm_narrow, fs=WORK_FS, ax=axes[0], title="Narrowband FM Spectrum", color="tab:orange")
plot_spectrum(fm_wide, fs=WORK_FS, ax=axes[1], title="Wideband FM Spectrum", color="tab:red")
for ax in axes:
    ax.set_xlim(0, 35_000)
    ax.set_ylim(-100, 5)
plt.tight_layout()

display(Markdown("**Narrowband FM demodulated audio**"))
display(audio_player(resample_signal(demod_narrow, WORK_FS, PLAY_FS), rate=PLAY_FS))
display(Markdown("**Wideband FM demodulated audio**"))
display(audio_player(resample_signal(demod_wide, WORK_FS, PLAY_FS), rate=PLAY_FS))


In [ ]:
audio_out = audio_output_widget()
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))

def update_deviation(freq_dev=2500.0):
    fm_sig = fm_modulate(message, carrier_freq=carrier_freq, fs=WORK_FS, freq_dev=freq_dev)
    demod = fm_demodulate(fm_sig, fs=WORK_FS)
    axes[0].clear()
    axes[1].clear()
    plot_waveform(fm_sig[:8000], fs=WORK_FS, ax=axes[0], title=f"Waveform, dev={freq_dev:.0f} Hz")
    plot_spectrum(fm_sig, fs=WORK_FS, ax=axes[1], title="Spectrum")
    axes[1].set_xlim(0, 35_000)
    axes[1].set_ylim(-100, 5)
    fig.canvas.draw_idle()
    refresh_audio_widget(audio_out, resample_signal(demod, WORK_FS, PLAY_FS), rate=PLAY_FS)

controls = widgets.interactive(
    update_deviation,
    freq_dev=float_slider(min_value=1500, max_value=6000, step=100, value=2500, description="Dev Hz"),
)
display(controls, audio_out)


## Key Takeaway

Deviation is a spectrum budget decision. More deviation generally buys better audio and better FM noise performance, but it costs occupied bandwidth.